In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0
from scipy.signal import freqz
from ipywidgets import Dropdown, FloatSlider, IntSlider, HBox, VBox, HTML, Layout
from IPython.display import display

# ============================================================
# KAISER WINDOW METHOD - INTERACTIVE THEORY NOTEBOOK
# FAST VERSION WITH ONE PRE-CREATED 2x3 FIGURE
# ============================================================

plt.ioff()

# ============================================================
# DISPLAY CONSTANTS
# ============================================================

TITLE_SIZE = 11.5
LABEL_SIZE = 10.5
TICK_SIZE = 9.5
LEGEND_SIZE = 9.0

UPPER_TEXT_SIZE = 14.0
UPPER_TITLE_SIZE = 14.0
WIDGET_FONT_SIZE = 13.5
INFO_FONT_SIZE = 13.5

NFFT_WINDOW = 4096
NFFT_FILTER = 2048

# ============================================================
# CSS
# ============================================================

style_html = HTML(f"""
<style>

.kw-root {{
    width:970px;
    max-width:970px;
    font-family:Arial,sans-serif;
}}

.kw-header {{
    background:linear-gradient(90deg,#0b5d70,#1597a8);
    color:white;
    padding:10px 15px;
    border-radius:8px 8px 0 0;
    font-size:{UPPER_TITLE_SIZE}px;
    font-weight:bold;
}}

.kw-intro {{
    background:#f4fbfc;
    border:1px solid #b7dce1;
    border-top:none;
    padding:10px 13px;
    border-radius:0 0 8px 8px;
    font-size:{UPPER_TEXT_SIZE}px;
    line-height:1.55;
    margin-bottom:8px;
}}

.kw-accent {{
    font-weight:bold;
    color:#0b6575;
}}

.kw-title {{
    font-size:{UPPER_TITLE_SIZE}px;
    font-weight:bold;
    margin:0 0 7px 2px;
    color:#0b6575;
}}

.kw-info {{
    font-size:{INFO_FONT_SIZE}px;
    line-height:1.55;
}}

.kw-value {{
    font-weight:bold;
}}

.widget-label {{
    font-size:{WIDGET_FONT_SIZE}px !important;
}}

.widget-readout {{
    font-size:{WIDGET_FONT_SIZE}px !important;
}}

.widget-dropdown > select {{
    font-size:{WIDGET_FONT_SIZE}px !important;
}}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {{
    overflow-x:visible !important;
    max-width:none !important;
}}

</style>
""")

# ============================================================
# NOTEBOOK DOCUMENTATION
# ============================================================

header_html = HTML("""
<div class="kw-root">

<div class="kw-header">
The Kaiser Window Method for FIR Filter Design
</div>

<div class="kw-intro">

<span class="kw-accent">Purpose.</span>
This notebook demonstrates the two fundamental design parameters of the Kaiser-window method.
The parameter <b>α</b> mainly controls the shape of the window and its sidelobe attenuation,
whereas the FIR order <b>N</b> mainly controls the width of the transition region.
<br><br>

<span class="kw-accent">Specifications → Kaiser.</span>
The user specifies Ã<sub>p</sub>, Ã<sub>s</sub>, ω<sub>p</sub> and ω<sub>s</sub>.
The design sequence is
δ̃<sub>p</sub>, δ̃<sub>s</sub> → δ → A<sub>s</sub> → α → D → N,
with ω<sub>c</sub>=(ω<sub>p</sub>+ω<sub>s</sub>)/2.
Since δ=min(δ̃<sub>p</sub>,δ̃<sub>s</sub>), only the stricter of the passband and stopband specifications controls the Kaiser parameters at any given time.
<br><br>

<span class="kw-accent">Explore α and N.</span>
The theoretical design remains available as a reference, but α and N can be varied directly.
This mode shows independently how α changes the window spectrum and how N affects the FIR response and delay.
<br><br>

<span class="kw-accent">Order convention.</span>
N denotes the FIR filter order. Therefore the impulse response contains N+1 samples and its generalized-linear-phase delay is N/2 samples.
When no sampling-frequency information is available, Ω<sub>s</sub>=2π is used.

</div>

</div>
""")

# ============================================================
# KAISER DESIGN EQUATIONS
# ============================================================

def delta_p_from_Ap(Ap):
    g = 10.0**(0.05*Ap)
    return (g-1.0)/(g+1.0)

def delta_s_from_As(As_tilde):
    return 10.0**(-0.05*As_tilde)

def actual_attenuation(delta):
    return -20.0*np.log10(delta)

def kaiser_alpha(As):
    if As <= 21.0:
        return 0.0
    if As <= 50.0:
        return 0.5842*(As-21.0)**0.4 + 0.07886*(As-21.0)
    return 0.1102*(As-8.7)

def kaiser_D(As):
    if As <= 21.0:
        return 0.9222
    return (As-7.95)/14.36

def kaiser_order(D, transition_width):
    return max(1,1+int((2.0*np.pi*D)/transition_width))

# ============================================================
# KAISER WINDOW
# ============================================================

def kaiser_window_from_order(N, alpha):
    n = np.arange(N+1,dtype=float)

    if N == 0:
        return n,np.ones(1)

    center = N/2.0
    x = (n-center)/center
    inside = np.maximum(0.0,1.0-x**2)
    w = i0(alpha*np.sqrt(inside))/i0(alpha)

    return n,w

# ============================================================
# IDEAL LOW-PASS IMPULSE RESPONSE
# ============================================================

def ideal_lowpass_impulse(N, wc):
    n = np.arange(N+1,dtype=float)
    m = n-N/2.0
    hd = np.empty_like(m)

    center = np.abs(m)<1e-12

    hd[center] = wc/np.pi
    hd[~center] = np.sin(wc*m[~center])/(np.pi*m[~center])

    return n,hd

# ============================================================
# ACTUAL FIR FILTER
# ============================================================

def construct_filter(N, alpha, wc):
    n,w = kaiser_window_from_order(N,alpha)
    _,hd = ideal_lowpass_impulse(N,wc)
    h = hd*w

    return n,w,hd,h

# ============================================================
# KAISER WINDOW SPECTRUM
# ============================================================

def calculate_window_spectrum(w):
    W = np.fft.fftshift(np.fft.fft(w,NFFT_WINDOW))
    omega = np.linspace(-1.0,1.0,NFFT_WINDOW,endpoint=False)

    magnitude = np.abs(W)
    magnitude /= max(np.max(magnitude),1e-15)

    Wdb = 20.0*np.log10(np.maximum(magnitude,1e-8))

    return omega,Wdb

# ============================================================
# FIR FREQUENCY RESPONSE
# ============================================================

def calculate_filter_response(h):
    omega,H = freqz(h,worN=NFFT_FILTER)

    Hmag = np.abs(H)
    phase = np.unwrap(np.angle(H))

    return omega/np.pi,Hmag,phase

# ============================================================
# CONTROLS
# ============================================================

mode_selector = Dropdown(options=['Specifications → Kaiser','Explore α and N'],value='Specifications → Kaiser',description='Mode:',style={'description_width':'55px'},layout=Layout(width='310px'))

Ap_slider = FloatSlider(value=0.10,min=0.02,max=3.00,step=0.02,description='Ãp [dB]:',continuous_update=True,readout_format='.2f',style={'description_width':'75px'},layout=Layout(width='305px'))

As_slider = FloatSlider(value=60.0,min=10.0,max=80.0,step=1.0,description='Ãs [dB]:',continuous_update=True,readout_format='.0f',style={'description_width':'75px'},layout=Layout(width='305px'))

wp_slider = FloatSlider(value=0.40,min=0.05,max=0.90,step=0.01,description='ωp / π:',continuous_update=True,readout_format='.2f',style={'description_width':'65px'},layout=Layout(width='305px'))

ws_slider = FloatSlider(value=0.60,min=0.10,max=0.95,step=0.01,description='ωs / π:',continuous_update=True,readout_format='.2f',style={'description_width':'65px'},layout=Layout(width='305px'))

alpha_slider = FloatSlider(value=5.65,min=0.0,max=12.0,step=0.05,description='α:',continuous_update=True,readout_format='.2f',style={'description_width':'40px'},layout=Layout(width='305px'))

N_slider = IntSlider(value=37,min=5,max=250,step=1,description='N:',continuous_update=True,style={'description_width':'40px'},layout=Layout(width='305px'))

# ============================================================
# INFORMATION PANEL
# ============================================================

info_html = HTML(layout=Layout(width='970px'))

def update_information(Ap, As_tilde, wp_norm, ws_norm, delta_p, delta_s, delta, As_actual, alpha_design, D, N_design, alpha_active, N_active):

    active_constraint = 'Passband (Ãp)' if delta_p <= delta_s else 'Stopband (Ãs)'

    info_html.value = f"""
    <div style="
        width:948px;
        border:1px solid #b7dce1;
        border-radius:6px;
        padding:11px 12px;
        font-family:Arial,sans-serif;
        margin-bottom:10px;
        font-size:{INFO_FONT_SIZE}px;
        line-height:1.55;
    ">

        <div style="display:flex; gap:30px;">

            <div style="flex:1;">

                <div style="
                    font-size:{UPPER_TITLE_SIZE}px;
                    font-weight:bold;
                    color:#0b6575;
                    margin-bottom:7px;
                ">
                    INPUT SPECIFICATIONS
                </div>

                Ãp = <b>{Ap:.2f} dB</b><br>
                Ãs = <b>{As_tilde:.2f} dB</b><br>
                ωp = <b>{wp_norm:.3f}π</b><br>
                ωs = <b>{ws_norm:.3f}π</b>

            </div>

            <div style="flex:1;">

                <div style="
                    font-size:{UPPER_TITLE_SIZE}px;
                    font-weight:bold;
                    color:#2874a6;
                    margin-bottom:7px;
                ">
                    KAISER CALCULATIONS
                </div>

                δ̃p = <b>{delta_p:.6f}</b><br>
                δ̃s = <b>{delta_s:.6f}</b><br>
                δ = <b>{delta:.6f}</b><br>
                As = <b>{As_actual:.3f} dB</b><br>
                D = <b>{D:.6f}</b><br>
                Active constraint = <b>{active_constraint}</b>

            </div>

            <div style="flex:1;">

                <div style="
                    font-size:{UPPER_TITLE_SIZE}px;
                    font-weight:bold;
                    color:#117864;
                    margin-bottom:7px;
                ">
                    DESIGN / ACTIVE PARAMETERS
                </div>

                Calculated α = <b>{alpha_design:.6f}</b><br>
                Calculated N = <b>{N_design}</b><br>
                Active α = <b>{alpha_active:.6f}</b><br>
                Active N = <b>{N_active}</b><br>
                Delay = <b>{N_active/2:.2f} samples</b>

            </div>

        </div>

    </div>
    """

# ============================================================
# INITIAL DESIGN
# ============================================================

wp0 = wp_slider.value*np.pi
ws0 = ws_slider.value*np.pi
wc0 = 0.5*(wp0+ws0)

delta_p0 = delta_p_from_Ap(Ap_slider.value)
delta_s0 = delta_s_from_As(As_slider.value)
delta0 = min(delta_p0,delta_s0)

As0 = actual_attenuation(delta0)
alpha0 = kaiser_alpha(As0)
D0 = kaiser_D(As0)
N0 = kaiser_order(D0,ws0-wp0)

n0,w0,hd0,h0 = construct_filter(N0,alpha0,wc0)
omegaW0,Wdb0 = calculate_window_spectrum(w0)
omegaH0,Hmag0,phase0 = calculate_filter_response(h0)

# ============================================================
# ONE FIGURE - SIX IDENTICAL AXES
# ============================================================

fig,axes = plt.subplots(2,3,figsize=(13.3,6.8))

fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.resizable = False

fig.canvas.layout = Layout(width='1200px',height='620px')

ax_window = axes[0,0]
ax_impulse = axes[0,1]
ax_mag = axes[0,2]

ax_window_spec = axes[1,0]
ax_group = axes[1,1]
ax_phase = axes[1,2]

# ============================================================
# COMMON AXIS FORMAT
# ============================================================

for ax in axes.flat:
    ax.tick_params(axis='both',labelsize=TICK_SIZE)
    ax.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# 1. KAISER WINDOW
# ============================================================

line_window, = ax_window.plot(n0,w0,'o-',color='red',linewidth=1.35,markersize=3.0)

ax_window.set_title('Kaiser Window $w[n]$',fontsize=TITLE_SIZE,pad=7)
ax_window.set_xlabel('Sample index $n$',fontsize=LABEL_SIZE)
ax_window.set_ylabel('$w[n]$',fontsize=LABEL_SIZE)
ax_window.set_ylim(-0.05,1.08)

# ============================================================
# 2. FIR IMPULSE RESPONSE
# ============================================================

line_impulse, = ax_impulse.plot(n0,h0,'o-',color='red',linewidth=1.35,markersize=3.0)

delay_line_impulse = ax_impulse.axvline(N0/2.0,linestyle='--',linewidth=1.1)

ax_impulse.set_title('Resulting FIR Impulse Response',fontsize=TITLE_SIZE,pad=7)
ax_impulse.set_xlabel('Sample index $n$',fontsize=LABEL_SIZE)
ax_impulse.set_ylabel('$h[n]$',fontsize=LABEL_SIZE)

# ============================================================
# 3. MAGNITUDE RESPONSE
# ============================================================

line_mag, = ax_mag.plot(omegaH0,Hmag0,color='red',linewidth=1.6,label='Actual Kaiser FIR')

line_ideal, = ax_mag.plot([0.0,wc0/np.pi,wc0/np.pi,1.0],[1.0,1.0,0.0,0.0],linestyle='--',linewidth=1.15,label='Ideal low-pass')

wp_line = ax_mag.axvline(wp_slider.value,linestyle=':',linewidth=1.0,label=r'$\omega_p$')
ws_line = ax_mag.axvline(ws_slider.value,linestyle=':',linewidth=1.0,label=r'$\omega_s$')
wc_line = ax_mag.axvline(wc0/np.pi,linestyle='-.',linewidth=1.0,label=r'$\omega_c$')

transition_patch = ax_mag.axvspan(wp_slider.value,ws_slider.value,alpha=0.10)

ax_mag.set_title('Ideal and Actual Magnitude Response',fontsize=TITLE_SIZE,pad=7)
ax_mag.set_xlabel(r'Normalized frequency $\omega/\pi$',fontsize=LABEL_SIZE)
ax_mag.set_ylabel(r'$|H(e^{j\omega})|$',fontsize=LABEL_SIZE)
ax_mag.set_xlim(0.0,1.0)
ax_mag.set_ylim(-0.08,1.16)

ax_mag.legend(loc='center left',bbox_to_anchor=(1.03,0.50),ncol=1,frameon=False,fontsize=LEGEND_SIZE,borderaxespad=0.0)

# ============================================================
# 4. WINDOW SPECTRUM
# ============================================================

line_window_spec, = ax_window_spec.plot(omegaW0,Wdb0,color='red',linewidth=1.35)

ax_window_spec.set_title('Kaiser Window Spectrum',fontsize=TITLE_SIZE,pad=7)
ax_window_spec.set_xlabel(r'Normalized frequency $\omega/\pi$',fontsize=LABEL_SIZE)
ax_window_spec.set_ylabel('Magnitude [dB]',fontsize=LABEL_SIZE)
ax_window_spec.set_xlim(-1.0,1.0)
ax_window_spec.set_ylim(-120,5)

# ============================================================
# 5. GROUP DELAY
# ============================================================

omega_group = np.linspace(0.0,1.0,500)

line_group, = ax_group.plot(omega_group,np.full_like(omega_group,N0/2.0),color='red',linewidth=1.5)

ax_group.set_title('Group Delay',fontsize=TITLE_SIZE,pad=7)
ax_group.set_xlabel(r'Normalized frequency $\omega/\pi$',fontsize=LABEL_SIZE)
ax_group.set_ylabel('Samples',fontsize=LABEL_SIZE)
ax_group.set_xlim(0.0,1.0)

# ============================================================
# 6. PHASE RESPONSE
# ============================================================

valid0 = Hmag0>1e-4

line_phase, = ax_phase.plot(omegaH0[valid0],phase0[valid0],color='red',linewidth=1.5)

ax_phase.set_title('Phase Response',fontsize=TITLE_SIZE,pad=7)
ax_phase.set_xlabel(r'Normalized frequency $\omega/\pi$',fontsize=LABEL_SIZE)
ax_phase.set_ylabel('Phase [rad]',fontsize=LABEL_SIZE)
ax_phase.set_xlim(0.0,1.0)

# ============================================================
# GRID SPACING
# ============================================================

fig.subplots_adjust(left=0.060,right=0.865,top=0.955,bottom=0.105,wspace=0.34,hspace=0.52)

# ============================================================
# FAST UPDATE FUNCTION
# ============================================================

def update_all(change=None):
    global transition_patch

    Ap = Ap_slider.value
    As_tilde = As_slider.value

    wp_norm = wp_slider.value
    ws_norm = ws_slider.value

    wp = wp_norm*np.pi
    ws = ws_norm*np.pi

    wc = 0.5*(wp+ws)
    transition_width = ws-wp

    delta_p = delta_p_from_Ap(Ap)
    delta_s = delta_s_from_As(As_tilde)

    delta = min(delta_p,delta_s)

    As_actual = actual_attenuation(delta)
    alpha_design = kaiser_alpha(As_actual)
    D = kaiser_D(As_actual)
    N_design = kaiser_order(D,transition_width)

    if mode_selector.value == 'Specifications → Kaiser':
        alpha = alpha_design
        N = N_design
    else:
        alpha = alpha_slider.value
        N = N_slider.value

    n,w,hd,h = construct_filter(N,alpha,wc)
    omegaW,Wdb = calculate_window_spectrum(w)
    omegaH,Hmag,phase = calculate_filter_response(h)

    # Kaiser window
    line_window.set_data(n,w)
    ax_window.set_xlim(-1,max(N+1,6))

    # FIR impulse response
    line_impulse.set_data(n,h)

    ymax = max(np.max(np.abs(h))*1.15,0.05)

    ax_impulse.set_xlim(-1,max(N+1,6))
    ax_impulse.set_ylim(-ymax,ymax)

    delay_line_impulse.set_xdata([N/2.0,N/2.0])

    # Magnitude response
    line_mag.set_data(omegaH,Hmag)

    line_ideal.set_data([0.0,wc/np.pi,wc/np.pi,1.0],[1.0,1.0,0.0,0.0])

    wp_line.set_xdata([wp_norm,wp_norm])
    ws_line.set_xdata([ws_norm,ws_norm])
    wc_line.set_xdata([wc/np.pi,wc/np.pi])

    transition_patch.remove()
    transition_patch = ax_mag.axvspan(wp_norm,ws_norm,alpha=0.10)

    # Window spectrum
    line_window_spec.set_data(omegaW,Wdb)

    # Group delay
    line_group.set_ydata(np.full_like(omega_group,N/2.0))

    ax_group.set_ylim(0.0,max(5.0,1.25*N/2.0))

    # Phase response
    valid = Hmag>1e-4

    line_phase.set_data(omegaH[valid],phase[valid])

    if np.any(valid):
        pmin = np.min(phase[valid])
        pmax = np.max(phase[valid])
        margin = max(0.5,0.05*(pmax-pmin))

        ax_phase.set_ylim(pmin-margin,pmax+margin)

    # Information panel
    update_information(Ap,As_tilde,wp_norm,ws_norm,delta_p,delta_s,delta,As_actual,alpha_design,D,N_design,alpha,N)

    fig.canvas.draw_idle()

# ============================================================
# MODE LOGIC
# ============================================================

def update_mode(change=None):
    manual_mode = mode_selector.value == 'Explore α and N'

    alpha_slider.disabled = not manual_mode
    N_slider.disabled = not manual_mode

    update_all()

# ============================================================
# FREQUENCY LIMITS
# ============================================================

def update_frequency_limits(change=None):
    gap = 0.01

    wp_slider.max = max(wp_slider.min,ws_slider.value-gap)
    ws_slider.min = min(ws_slider.max,wp_slider.value+gap)

# ============================================================
# OBSERVERS
# ============================================================

mode_selector.observe(update_mode,names='value')

Ap_slider.observe(update_all,names='value')
As_slider.observe(update_all,names='value')

wp_slider.observe(update_frequency_limits,names='value')
ws_slider.observe(update_frequency_limits,names='value')

wp_slider.observe(update_all,names='value')
ws_slider.observe(update_all,names='value')

alpha_slider.observe(update_all,names='value')
N_slider.observe(update_all,names='value')

# ============================================================
# INITIAL STATE
# ============================================================

alpha_slider.disabled = True
N_slider.disabled = True

update_frequency_limits()
update_all()

# ============================================================
# CONTROL LAYOUT
# ============================================================

mode_box = VBox(
    [
        HTML("<div class='kw-title'>Operating Mode</div>"),
        HBox([mode_selector])
    ],
    layout=Layout(width='970px',border='1px solid #b7dce1',padding='9px 11px')
)

specification_box = VBox(
    [
        HTML("<div class='kw-title'>Filter Specifications</div>"),
        HBox([Ap_slider,As_slider],layout=Layout(width='630px',justify_content='space-between')),
        HBox([wp_slider,ws_slider],layout=Layout(width='630px',justify_content='space-between'))
    ],
    layout=Layout(width='970px',border='1px solid #b7dce1',padding='9px 11px')
)

exploration_box = VBox(
    [
        HTML("<div class='kw-title'>Direct Kaiser Parameter Exploration</div>"),
        HBox([alpha_slider,N_slider],layout=Layout(width='630px',justify_content='space-between'))
    ],
    layout=Layout(width='970px',border='1px solid #b7dce1',padding='9px 11px')
)

# ============================================================
# OUTPUT
# ============================================================

figure_box = VBox(
    [fig.canvas],
    layout=Layout(width='auto',overflow='visible',margin='8px 0 0 0')
)

main_layout = VBox(
    [
        header_html,
        mode_box,
        specification_box,
        exploration_box,
        info_html,
        figure_box
    ],
    layout=Layout(width='970px',overflow='visible',align_items='flex-start')
)

# ============================================================
# DISPLAY
# ============================================================

display(style_html)
display(main_layout)